# Moon-SoC Hierarchical Macro Synthesis & Top-Level Assembly in LibreLane

This notebook synthesizes and hardens **Moon-SoC** hierarchically into independent physical macros:
- **System Control (AON/CRG/PMU)** (`moon_sysctrl_subsystem`)
- **Peripheral Subsystem** (`moon_periph_subsystem`)
- **System Fabric** (`moon_sys_fabric`)
- **Memory Subsystem** (`moon_mem_subsystem`)
- **Cache Tile** (`moon_cache_tile`)
- **CPU Compute Tile** (`moon_cpu_tile`)

And then assembles the complete top-level SoC (`moon_soc_top`) with the I/O & Memory PHY boundary cells placed at the pad frame.

## 1. Setup Nix and LibreLane

In [ ]:
# @title Setup Nix & LibreLane {display-mode: "form"}
import os
from pathlib import Path
import subprocess
import sys
import shutil
import tempfile

os.environ["LOCALE_ARCHIVE"] = "/usr/lib/locale/locale-archive"

if "google.colab" in sys.modules:
    if shutil.which("nix-env") is None:
        with tempfile.TemporaryDirectory() as d:
            d = Path(d)
            installer_path = d / "nix"
            !curl --proto '=https' --tlsv1.2 -sSf -L https://install.determinate.systems/nix > {installer_path}
            with subprocess.Popen(
                [
                    "bash",
                    installer_path,
                    "install",
                    "--prefer-upstream-nix",
                    "--no-confirm",
                    "--extra-conf",
                    "extra-substituters = https://nix-cache.fossi-foundation.org\nextra-trusted-public-keys = nix-cache.fossi-foundation.org:3+K59iFwXqKsL7BNu6Guy0v+uTlwsxYQxjspXzqLYQs=\n",
                ],
                stdout=subprocess.PIPE,
                stderr=subprocess.STDOUT,
                encoding="utf8",
            ) as p:
                for line in p.stdout:
                    print(line, end="")
else:
    if shutil.which("nix-env") is None:
        raise RuntimeError("Nix is not installed!")

os.environ["PATH"] = f"/nix/var/nix/profiles/default/bin/:{os.getenv('PATH')}"

import os
import yaml
import subprocess
import IPython

librelane_version = "latest"  # @param {key:"LibreLane Version", type:"string"}

if librelane_version == "latest":
    librelane_version = "main"

pdk_root = "~/.ciel"  # @param {key:"PDK Root", type:"string"}

pdk_root = os.path.expanduser(pdk_root)

pdk = "sky130"  # @param {key:"PDK (without the variant)", type:"string"}

librelane_ipynb_path = os.path.join(os.getcwd(), "librelane_ipynb")

display(IPython.display.HTML("<h3>Downloading LibreLane…</a>"))


TESTING_LOCALLY = False
!rm -rf {librelane_ipynb_path}
!mkdir -p {librelane_ipynb_path}
if TESTING_LOCALLY:
    !ln -s {os.getcwd()} {librelane_ipynb_path}
else:
    !curl -L "https://github.com/librelane/librelane/tarball/{librelane_version}" | tar -xzC {librelane_ipynb_path} --strip-components 1

try:
    import tkinter
except ImportError:
    if "google.colab" in sys.modules:
        !sudo apt-get install python-tk

try:
    import tkinter
except ImportError as e:
    display(
        IPython.display.HTML(
            '<h3 style="color: #800020";>❌ Failed to import the <code>tkinter</code> library for Python, which is required to load PDK configuration values. Make sure <code>python3-tk</code> or equivalent is installed on your system.</a>'
        )
    )
    raise e from None


display(IPython.display.HTML("<h3>Downloading LibreLane's dependencies…</a>"))
try:
    with subprocess.Popen(
        [
            "nix",
            "profile",
            "install",
            ".#colab-env",
        ],
        cwd=librelane_ipynb_path,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        encoding="utf8",
    ) as p:
        for line in p.stdout:
            print(line, end="")
except subprocess.CalledProcessError as e:
    display(
        IPython.display.HTML(
            '<h3 style="color: #800020";>❌ Failed to install binary dependencies using Nix…</h3>'
        )
    )

display(IPython.display.HTML("<h3>Downloading Python dependencies using PIP…</a>"))
try:
    subprocess.check_call(
        ["pip3", "install", "."],
        cwd=librelane_ipynb_path,
    )
except subprocess.CalledProcessError as e:
    display(
        IPython.display.HTML(
            '<h3 style="color: #800020";>❌ Failed to install Python dependencies using PIP…</h3>'
        )
    )
    raise e from None

display(IPython.display.HTML("<h3>Downloading PDK…</a>"))
import ciel
from ciel.source import StaticWebDataSource

with open(
    os.path.join(librelane_ipynb_path, "librelane", "pdk_hashes.yaml"), "r"
) as file:
    pdk_hashes = yaml.safe_load(file)

ciel.enable(
    ciel.get_ciel_home(pdk_root),
    pdk,
    pdk_hashes[pdk],
    data_source=StaticWebDataSource("https://fossi-foundation.github.io/ciel-releases"),
)

sys.path.insert(0, librelane_ipynb_path)
display(IPython.display.HTML("<h3>⭕️ Done.</a>"))

import logging

# Remove the stupid default colab logging handler
logging.getLogger().handlers.clear()

## 2. Clone / Locate Moon-SoC Repository

In [ ]:
repo_path = "/content/moon-soc" if "google.colab" in sys.modules else os.getcwd()
if not os.path.exists(repo_path) and "google.colab" in sys.modules:
    !git clone https://github.com/phoshp/moon-soc.git {repo_path}

synth_dir = os.path.join(repo_path, "synth")
print(f"Using synthesis directory: {synth_dir}")

## 3. Harden Macro Partitions Separately

We harden each functional subsystem individually. Each block produces `.lef`, `.lib`, and `.gds` files in its `runs/latest/final/` directory.

In [ ]:
from librelane.flows import Flow

macros = [
    ("System Control (AON/CRG/PMU)", "synth/macros/sysctrl_subsystem"),
    ("Peripheral Subsystem",         "synth/macros/periph_subsystem"),
    ("System Fabric",                "synth/macros/sys_fabric"),
    ("Memory Subsystem",             "synth/macros/mem_subsystem"),
    ("Cache Tile",                   "synth/macros/cache_tile"),
    ("CPU Compute Tile",             "synth/macros/cpu_tile")
]

for label, rel_dir in macros:
    config_file = os.path.join(repo_path, rel_dir, "config.json")
    print(f"\n{'='*70}\n>>> Hardening Macro: {label} ({rel_dir})\n{'='*70}")
    flow = Flow.factory.get("Classic")(config_file)
    flow.start()
    print(f"[DONE] Macro {label} hardened successfully!")

## 4. Assemble Top-Level Moon-SoC Chip

Now synthesize and route the top-level `moon_soc_top`. Yosys reads the blackbox macro stubs and elaborates the inter-macro connections in **seconds**! OpenROAD then places the hardened macro LEFs according to `macro_placement.cfg` and routes the top-level chip.

In [ ]:
top_config = os.path.join(synth_dir, "top", "config.json")
print(f"Assembling Top-Level SoC with Macro Placement from: {top_config}")

flow_top = Flow.factory.get("Classic")(config_file=top_config)
flow_top.start()
print("\n[SUCCESS] Moon-SoC Top-Level Chip implementation completed!")

## 5. Inspect Final Layout (GDSII / KLayout)

In [ ]:
top_run = os.path.join(synth_dir, "top", "runs", "latest", "final", "gds", "moon_soc_top.gds")
if os.path.exists(top_run):
    print(f"Top-level GDSII available at: {top_run}")
    print(f"Size: {os.path.getsize(top_run) / (1024*1024):.2f} MB")
else:
    print("Run output will be placed in runs/latest/final/gds/")